# Imidazolone & Thiazolone Library Generation

Virtual combinatorial library of COX-2 selective inhibitors derived from 
commercially available aldehydes, carboxylic acids and amines via 
reactions *in silico*.

**Pipeline:** Aldehydes + Carboxylic Acids → Erlenmeyer-Plöchl → Oxazolones → Aminolysis-GFPc → Imidazolones  
**Amines** feed into the Aminolysis-GFPc step as co-reactants.
**Parallel branch:** Oxazolones → Sulphur-Exchange → Thiazolones  


All three building-block sets pass Price → Veber filters before reactions.  
Final products (Imidazolones and Thiazolones) are filtered by Veber and Brenk+PAINS before export,  
and saved as **separate** CSV files.


In [ ]:
from rdkit import Chem
import numpy as np
import pandas as pd

import py_utils as pu

print(f"py_utils v{pu.__version__}")

## 📥 1. Load SDFs

The SDFs of the compounds provided by Enamine (with their IDs) that match a
specific substructure are loaded:
- Aldehydes, R-CHO
- Carboxylic compounds, R-COOH or RCOX
- Primary amines, R-NH₂

In [ ]:
sdf_path_aldehydes   = 'mol_files/1. Enamine SDFs/TESTER_Aldehydes_120.sdf'
sdf_path_carboxylics = 'mol_files/1. Enamine SDFs/TESTER_CarboxylicAcids_750.sdf'
sdf_path_amines      = 'mol_files/1. Enamine SDFs/TESTER_PrimaryAmines_680.sdf'

df_aldehydes_raw   = pu.sdf_to_dataframe(sdf_path_aldehydes,   id_prefix='A')
df_carboxylics_raw = pu.sdf_to_dataframe(sdf_path_carboxylics, id_prefix='C')
df_amines_raw      = pu.sdf_to_dataframe(sdf_path_amines,      id_prefix='N') 

pu.report_df_size(df_aldehydes_raw,   'Aldehydes - Raw')
pu.report_df_size(df_carboxylics_raw, 'Carboxylics - Raw')
pu.report_df_size(df_amines_raw,      'Amines - Raw')

## 🔸 2. Price filtration

Query the Enamine Store API for real-time pricing.  Compounds exceeding the
price thresholds are discarded during retrieval (no wasted work).

In [ ]:
client = pu.EnamineClient()

print('=== Querying Enamine API for Aldehydes ===')
df_aldehydes_cheap = pu.add_enamine_prices(df_aldehydes_raw, client=client)

print('\n=== Querying Enamine API for Carboxylic Acids ===')
df_carboxylics_cheap = pu.add_enamine_prices(df_carboxylics_raw, client=client)

print('\n=== Querying Enamine API for Amines ===')
df_amines_cheap = pu.add_enamine_prices(df_amines_raw, client=client)

print('\n' + '='*50)
print('PRICING SUMMARY')
print('='*50)
pu.report_df_size(df_aldehydes_cheap,   'Aldehydes - Cheap')
pu.report_df_size(df_carboxylics_cheap, 'Carboxylics - Cheap')
pu.report_df_size(df_amines_cheap,      'Amines - Cheap')

## 🔸 3. Veber filtering (of building blocks)

Compute RDKit molecular descriptors, then apply Veber bioavailability criteria.
Adjust `VEBER_BB` limits as needed.

In [ ]:
VEBER_BB = dict(max_tPSA=90, max_RotB=10, max_LogP=3)

df_aldehydes_druglike, df_aldehydes_rej = pu.filter_Veber(
    pu.add_rdkit_properties(df_aldehydes_cheap), **VEBER_BB,
)
df_carboxylics_druglike, df_carboxylics_rej = pu.filter_Veber(
    pu.add_rdkit_properties(df_carboxylics_cheap), **VEBER_BB,
)
df_amines_druglike, df_amines_rej = pu.filter_Veber(
    pu.add_rdkit_properties(df_amines_cheap), **VEBER_BB,
)

pu.report_df_size(df_aldehydes_druglike,   'Aldehydes - Druglike')
pu.report_df_size(df_carboxylics_druglike, 'Carboxylics - Druglike')
pu.report_df_size(df_amines_druglike,      'Amines - Druglike')

# Save Veber-filtered building blocks
pu.save_dataframe_as_csv(df_aldehydes_druglike,   'mol_files/2. Building Blocks/Aldehydes_druglike')
pu.save_dataframe_as_csv(df_aldehydes_rej,        'mol_files/2. Building Blocks/Rejected/Aldehydes_rejected_veber')
pu.save_dataframe_as_csv(df_carboxylics_druglike, 'mol_files/2. Building Blocks/Carboxylics_druglike')
pu.save_dataframe_as_csv(df_carboxylics_rej,      'mol_files/2. Building Blocks/Rejected/Carboxylics_rejected_veber')
pu.save_dataframe_as_csv(df_amines_druglike,      'mol_files/2. Building Blocks/Amines_druglike')
pu.save_dataframe_as_csv(df_amines_rej,           'mol_files/2. Building Blocks/Rejected/Amines_rejected_veber')

## 🔸 4. Erlenmeyer-Plöchl reaction

Aldehydes + Carboxylic Acids → Oxazolones

In [ ]:
df_oxazolones_raw = pu.rxn_ErlenmeyerPlochl(df_aldehydes_druglike, df_carboxylics_druglike)

pu.report_df_size(df_oxazolones_raw, 'Oxazolones - Raw')

## 🔸 5. Veber filtering (of intermediates)

Compute descriptors and apply Veber criteria to oxazolones.

In [ ]:
df_oxazolones_druglike, df_oxazolones_rej = pu.filter_Veber(
    pu.add_rdkit_properties(df_oxazolones_raw),
    max_tPSA=90, max_RotB=10, max_LogP=3,  # TODO: adjust for intermediates if needed
)

pu.report_df_size(df_oxazolones_druglike, 'Oxazolones - Druglike')

# Save intermediates
pu.save_dataframe_as_csv(df_oxazolones_druglike, 'mol_files/3. Oxazolones/Oxazolones')
pu.save_dataframe_as_csv(df_oxazolones_rej,      'mol_files/3. Oxazolones/Rejected/Oxazolones_rejected_veber')

## 🔹 6. Aminolysis-GFPc reaction

Oxazolones + Amines → Imidazolones

In [ ]:
df_imidazolones_raw = pu.rxn_AminolysisGFPc(df_oxazolones_druglike, df_amines_druglike)

pu.report_df_size(df_imidazolones_raw, 'Imidazolones - Raw')

## 🔹 7. Sulphur-Exchange reaction

Oxazolones → Thiazolones

> **Note:** input is `df_oxazolones_druglike` (post-Veber, Cell 5) — not `df_imidazolones_raw` or `df_oxazolones_raw`.

In [ ]:
df_thiazolones_raw = pu.rxn_SulphurExchange(df_oxazolones_druglike)

pu.report_df_size(df_thiazolones_raw, 'Thiazolones - Raw')

## 🔸 8. Veber filtering (of products)

Compute descriptors and apply Veber criteria to both product families.
Adjust `VEBER_PRODUCTS` limits independently from building blocks.

In [ ]:
VEBER_PRODUCTS = dict(max_tPSA=90, max_RotB=10, max_LogP=3)  # TODO: adjust limits for products

df_imidazolones_druglike, df_imidazolones_rej = pu.filter_Veber(
    pu.add_rdkit_properties(df_imidazolones_raw), **VEBER_PRODUCTS,
)

df_thiazolones_druglike, df_thiazolones_rej = pu.filter_Veber(
    pu.add_rdkit_properties(df_thiazolones_raw), **VEBER_PRODUCTS,
)

pu.report_df_size(df_imidazolones_druglike, 'Imidazolones - Druglike')
pu.report_df_size(df_thiazolones_druglike,  'Thiazolones - Druglike')

## 🔸 9. Brenk + PAINS filters

Remove compounds containing Brenk structural alerts or PAINS substructures from products.


In [ ]:
df_imidazolones_untoxic, df_imidazolones_rej = pu.filter_BrenkPAINS(df_imidazolones_druglike)
df_thiazolones_untoxic, df_thiazolones_rej = pu.filter_BrenkPAINS(df_thiazolones_druglike)

pu.report_df_size(df_imidazolones_untoxic, 'Imidazolones - Untoxic')
pu.report_df_size(df_thiazolones_untoxic,  'Thiazolones - Untoxic')

# Save rejected for analysis
pu.save_dataframe_as_csv(df_imidazolones_rej, 'mol_files/4. Imidazolones/Rejected/Imidazolones_rejected_brenkpains')
pu.save_dataframe_as_csv(df_thiazolones_rej,  'mol_files/5. Thiazolones/Rejected/Thiazolones_rejected_brenkpains')


## 📤 10. Save CSVs

Imidazolones and Thiazolones saved as **separate** CSV files.

In [ ]:
# TODO: define columns
# df_imidazolones_untoxic and df_thiazolones_untoxic must be saved SEPARATELY
pu.save_dataframe_as_csv(df_imidazolones_untoxic, 'mol_files/4. Imidazolones/Imidazolones')
pu.save_dataframe_as_csv(df_thiazolones_untoxic,  'mol_files/5. Thiazolones/Thiazolones')